# inf-masking — worked example 1: Causal attention via -inf masking

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inf-masking`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Filling forbidden score entries with `-inf` before a softmax forces their post-softmax weight to be exactly zero, because `exp(-inf) = 0`. A causal mask blocks each query from attending to future keys; `masked_fill(mask, -inf)` then `softmax` is the standard implementation.

## Worked solution

We build causal attention weights from a `(T, T)` score matrix.

1. Construct the causal mask `t.triu(t.ones(T, T, dtype=t.bool), diagonal=1)` — `True` strictly above the diagonal marks future positions a query may not see.
2. `scores.masked_fill(mask, float('-inf'))` overwrites those entries with `-inf`. We use `-inf` rather than a large negative number so the masked weight is exactly zero, not merely tiny.
3. `softmax(dim=-1)` normalizes along the key axis. The `-inf` entries contribute `exp(-inf)=0`, so they vanish and the remaining entries renormalize to sum to 1.
4. We confirm the upper triangle is exactly zero and each row sums to 1.

In [ ]:
import torch as t

t.manual_seed(0)
scores = t.randn(4, 4)

def causal_softmax(scores):
    T = scores.shape[-1]
    mask = t.triu(t.ones(T, T, dtype=t.bool), diagonal=1)
    return scores.masked_fill(mask, float('-inf')).softmax(dim=-1)

w = causal_softmax(scores)
print('upper-tri zero:', bool((w.triu(diagonal=1) == 0).all()))
print('rows sum to 1:', bool(t.allclose(w.sum(-1), t.ones(4))))